# 第10章 代码教学：通用信息抽取（IE）与序列标注

本教程以**最小可运行实验**贯穿 IE 关键流程：从序列标注到 NER，再扩展到 RE 与文档字段抽取。

学习目标：
1) 掌握 BIO/BILOU 标签体系与子词对齐
2) 训练轻量 NER 并计算 F1
3) 基于 NER 结果做规则化 RE
4) 在票据/发票场景进行字段抽取与可视化

评价指标：Precision / Recall / F1（以 micro-F1 为主）。

说明：为保证可复现，使用极小样本与轻量模型；真实项目需扩充数据并严格评测。


## 0. 环境准备

依赖尽量少：`transformers` `datasets` `seqeval` `torch`。
如处在离线环境，请提前缓存模型或配置镜像。


In [1]:
# 如需安装/升级（可选）
# !pip install -U transformers datasets seqeval accelerate

import random
import re
import numpy as np
import torch
from datasets import Dataset
from seqeval.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cpu cuda False


## 1. 构造最小 IE 数据

我们用“合同/公告/票据”风格文本，实体类型包括：
- `ORG` 机构
- `DATE` 日期
- `MONEY` 金额
- `INVOICE_NO` 票据号
- `DOC_TYPE` 文档类型


In [2]:
# 每条样本包含 tokens（已分词）与实体跨度 (start, end, label)
raw_samples = [
    {
        'tokens': ['Vendor','BeijingTech','signed','contract','with','ShanghaiCloud','on','2024-03-12','for','1.28M','CNY','.'],
        'entities': [(1,2,'ORG'), (5,6,'ORG'), (7,8,'DATE'), (9,11,'MONEY')],
    },
    {
        'tokens': ['Announcement',':','ShenzhenAI','acquired','DataBridge','on','2025-01-15','for','3.2','billion','USD','.'],
        'entities': [(0,1,'DOC_TYPE'), (2,3,'ORG'), (4,5,'ORG'), (6,7,'DATE'), (8,11,'MONEY')],
    },
    {
        'tokens': ['Invoice','INV-2024-0312','issued','to','DeltaFactory','amount','98000','CNY','on','2024-03-20','.'],
        'entities': [(0,1,'DOC_TYPE'), (1,2,'INVOICE_NO'), (4,5,'ORG'), (6,8,'MONEY'), (9,10,'DATE')],
    },
    {
        'tokens': ['Contract','between','AlphaCompute','and','BetaStorage','effective','2024-07-01','.'],
        'entities': [(0,1,'DOC_TYPE'), (2,3,'ORG'), (4,5,'ORG'), (6,7,'DATE')],
    },
    {
        'tokens': ['Invoice','INV-2024-0451','payer','NanoVision','total','120000','CNY','.'],
        'entities': [(0,1,'DOC_TYPE'), (1,2,'INVOICE_NO'), (3,4,'ORG'), (5,7,'MONEY')],
    },
]

print('samples =', len(raw_samples))
print(raw_samples[0])


samples = 5
{'tokens': ['Vendor', 'BeijingTech', 'signed', 'contract', 'with', 'ShanghaiCloud', 'on', '2024-03-12', 'for', '1.28M', 'CNY', '.'], 'entities': [(1, 2, 'ORG'), (5, 6, 'ORG'), (7, 8, 'DATE'), (9, 11, 'MONEY')]}


## 2. BIO / BILOU 标签体系

- BIO：B-开始，I-内部，O-非实体
- BILOU：B/I/L/U，更细粒度区分单 token 实体与结束位置

我们先转 BIO，再演示 BILOU。


In [3]:
def spans_to_bio(tokens, entities):
    tags = ['O'] * len(tokens)
    for s, e, label in entities:
        tags[s] = f'B-{label}'
        for i in range(s+1, e):
            tags[i] = f'I-{label}'
    return tags


def bio_to_bilou(tags):
    out = []
    for i, t in enumerate(tags):
        if t == 'O':
            out.append('O')
            continue
        p, label = t.split('-', 1)
        if p == 'B':
            single = (i+1 == len(tags)) or (not tags[i+1].startswith('I-'))
            out.append(f'U-{label}' if single else f'B-{label}')
        elif p == 'I':
            last = (i+1 == len(tags)) or (not tags[i+1].startswith('I-'))
            out.append(f'L-{label}' if last else f'I-{label}')
        else:
            out.append(t)
    return out


In [4]:
processed = []
for s in raw_samples:
    bio = spans_to_bio(s['tokens'], s['entities'])
    bilou = bio_to_bilou(bio)
    processed.append({'tokens': s['tokens'], 'bio': bio, 'bilou': bilou})

print(list(zip(processed[0]['tokens'], processed[0]['bio'])))
print(list(zip(processed[0]['tokens'], processed[0]['bilou'])))


[('Vendor', 'O'), ('BeijingTech', 'B-ORG'), ('signed', 'O'), ('contract', 'O'), ('with', 'O'), ('ShanghaiCloud', 'B-ORG'), ('on', 'O'), ('2024-03-12', 'B-DATE'), ('for', 'O'), ('1.28M', 'B-MONEY'), ('CNY', 'I-MONEY'), ('.', 'O')]
[('Vendor', 'O'), ('BeijingTech', 'U-ORG'), ('signed', 'O'), ('contract', 'O'), ('with', 'O'), ('ShanghaiCloud', 'U-ORG'), ('on', 'O'), ('2024-03-12', 'U-DATE'), ('for', 'O'), ('1.28M', 'B-MONEY'), ('CNY', 'L-MONEY'), ('.', 'O')]


## 3. 组装 Dataset + 标签映射

训练使用 BIO 标签（BILOU 可用于分析）。


In [5]:
random.shuffle(processed)
sp = int(len(processed) * 0.8)
train_samples, val_samples = processed[:sp], processed[sp:]

label_list = sorted({t for s in processed for t in s['bio']})
label2id = {t:i for i,t in enumerate(label_list)}
id2label = {i:t for t,i in label2id.items()}


def to_rows(samples):
    return [{
        'tokens': s['tokens'],
        'ner_tags': [label2id[t] for t in s['bio']]
    } for s in samples]

train_ds = Dataset.from_list(to_rows(train_samples))
val_ds = Dataset.from_list(to_rows(val_samples))

print('labels =', label_list)
print('train/val =', len(train_ds), len(val_ds))


labels = ['B-DATE', 'B-DOC_TYPE', 'B-INVOICE_NO', 'B-MONEY', 'B-ORG', 'I-MONEY', 'O']
train/val = 4 1


## 4. Tokenization + Label 对齐

子词切分后，非首子词与特殊 token 使用 `-100`，避免参与损失。


In [6]:
MODEL = 'prajjwal1/bert-tiny'  # 真实存在的轻量模型
# 如离线或下载失败，可改成已缓存模型名
# MODEL = 'distilbert-base-uncased'

tok = AutoTokenizer.from_pretrained(MODEL)


def align(batch):
    z = tok(batch['tokens'], is_split_into_words=True, truncation=True)
    labels = []
    for i, lab in enumerate(batch['ner_tags']):
        word_ids = z.word_ids(batch_index=i)
        cur = []
        prev = None
        for wid in word_ids:
            if wid is None:
                cur.append(-100)
            elif wid != prev:
                cur.append(lab[wid])
            else:
                cur.append(-100)
            prev = wid
        labels.append(cur)
    z['labels'] = labels
    return z

train_tok = train_ds.map(align, batched=True)
val_tok = val_ds.map(align, batched=True)
print(train_tok[0].keys())


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map: 100%|██████████| 4/4 [00:00<00:00, 825.85 examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map: 100%|██████████| 1/1 [00:00<00:00, 460.51 examples/s]

dict_keys(['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])


## 5. 训练轻量 NER 模型


In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(label_list), id2label=id2label, label2id=label2id
)
collator = DataCollatorForTokenClassification(tok)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    y_pred, y_true = [], []
    for p_seq, l_seq in zip(preds, labels):
        cur_p, cur_t = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100:
                continue
            cur_p.append(id2label[int(p)])
            cur_t.append(id2label[int(l)])
        y_pred.append(cur_p)
        y_true.append(cur_t)
    return {'f1': f1_score(y_true, y_pred)}

import inspect

ta_kwargs = dict(
    output_dir='./tmp/ch10_ie',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,
    save_strategy='no',
    logging_strategy='epoch',
    report_to='none',
    fp16=torch.cuda.is_available(),
)
sig = inspect.signature(TrainingArguments.__init__).parameters
if 'evaluation_strategy' in sig:
    ta_kwargs['evaluation_strategy'] = 'epoch'
else:
    ta_kwargs['eval_strategy'] = 'epoch'

args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())


Loading weights:   0%|          | 0/37 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 1/37 [00:00<00:00, 27060.03it/s, Materializing param=bert.embeddings.LayerNorm.bias]

Loading weights:   3%|▎         | 1/37 [00:00<00:00, 3912.60it/s, Materializing param=bert.embeddings.LayerNorm.bias] 

Loading weights:   5%|▌         | 2/37 [00:00<00:00, 3729.93it/s, Materializing param=bert.embeddings.LayerNorm.weight]

Loading weights:   5%|▌         | 2/37 [00:00<00:00, 2781.37it/s, Materializing param=bert.embeddings.LayerNorm.weight]

...

Loading weights: 100%|██████████| 37/37 [00:00<00:00, 2404.25it/s, Materializing param=bert.encoder.layer.1.output.dense.weight]


BertForTokenClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISS

C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,1.927221,1.739946,0.000000
2,1.822093,1.709814,0.000000
3,1.827255,1.685034,0.000000
4,1.812631,1.665586,0.000000
5,1.791530,1.650548,0.000000
6,1.788735,1.639599,0.000000
7,1.794875,1.632479,0.000000
8,1.750302,1.628936,0.000000


{'eval_loss': 1.628935694694519, 'eval_f1': 0.0, 'eval_runtime': 0.0044, 'eval_samples_per_second': 229.586, 'eval_steps_per_second': 229.586, 'epoch': 8.0}


## 6. NER 推理与结果可视化


In [8]:
ner = pipeline('token-classification', model=model, tokenizer=tok, aggregation_strategy='simple')
text = 'Contract between BeijingTech and ShanghaiCloud on 2025-03-01 for 450000 CNY.'
ents = ner(text)
print('text:', text)
for e in ents:
    print(e)


text: Contract between BeijingTech and ShanghaiCloud on 2025-03-01 for 450000 CNY.


## 7. 基于 NER 的关系抽取（RE）

这里用最小规则演示（便于理解链路）。生产中可替换为 RE 模型或 LLM 指令抽取。


In [9]:
def extract_relations(txt, ents):
    orgs = [e for e in ents if e.get('entity_group') == 'ORG']
    dates = [e for e in ents if e.get('entity_group') == 'DATE']
    moneys = [e for e in ents if e.get('entity_group') == 'MONEY']
    rel = []
    if 'contract' in txt.lower() and len(orgs) >= 2:
        rel.append({'head': orgs[0]['word'], 'rel': 'CONTRACT_WITH', 'tail': orgs[1]['word']})
    if dates and moneys:
        rel.append({'head': moneys[0]['word'], 'rel': 'EFFECTIVE_ON', 'tail': dates[0]['word']})
    return rel

print(extract_relations(text, ents))


[]


## 8. 文档字段抽取（票据/发票）

演示“规则抽取”作为基线，真实项目可升级为 LLM 结构化抽取。


In [10]:
invoice = 'Invoice No: INV-2025-2331; Buyer: NanoVision; Amount: 268000 CNY; Date: 2025-04-19'

def parse_invoice(t):
    return {
        'invoice_no': (re.search(r'INV-\d{4}-\d{4}', t).group(0) if re.search(r'INV-\d{4}-\d{4}', t) else None),
        'buyer': (re.search(r'Buyer:\s*([A-Za-z][A-Za-z0-9_-]+)', t).group(1) if re.search(r'Buyer:\s*([A-Za-z][A-Za-z0-9_-]+)', t) else None),
        'amount_cny': (float(re.search(r'(\d+(?:\.\d+)?)\s*CNY', t).group(1)) if re.search(r'(\d+(?:\.\d+)?)\s*CNY', t) else None),
        'date': (re.search(r'\d{4}-\d{2}-\d{2}', t).group(0) if re.search(r'\d{4}-\d{2}-\d{2}', t) else None),
    }

print(parse_invoice(invoice))


{'invoice_no': 'INV-2025-2331', 'buyer': 'NanoVision', 'amount_cny': 268000.0, 'date': '2025-04-19'}


## 9. 详细报告（可选）


In [11]:
# 如果你有更多验证样本，可输出更细粒度报告
preds = trainer.predict(val_tok)
pred_ids = np.argmax(preds.predictions, axis=-1)

y_true, y_pred = [], []
for p_seq, l_seq in zip(pred_ids, preds.label_ids):
    cur_t, cur_p = [], []
    for p, l in zip(p_seq, l_seq):
        if l == -100:
            continue
        cur_t.append(id2label[int(l)])
        cur_p.append(id2label[int(p)])
    y_true.append(cur_t)
    y_pred.append(cur_p)

print(classification_report(y_true, y_pred, digits=4))


              precision    recall  f1-score   support

        DATE     0.0000    0.0000    0.0000         1
       MONEY     0.0000    0.0000    0.0000         1
         ORG     0.0000    0.0000    0.0000         2

   micro avg     0.0000    0.0000    0.0000         4
   macro avg     0.0000    0.0000    0.0000         4
weighted avg     0.0000    0.0000    0.0000         4



C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## 10. 练习
1) 扩展事件触发词标签并构建事件抽取数据。
2) 用更大模型对比 F1 与吞吐。
3) 接入 OCR 文本做票据字段抽取。
